# TIDES: TikTok influencer tiers and engagement

[Paper](https://ojs.aaai.org/index.php/ICWSM/article/view/31405) · [Authors' dataset repository](https://github.com/skmtr1/TIDES-TikTok-Influencer-dataset). This notebook analyses only the TikTok tables and keeps video-level and creator-level summaries separate. `comments` is an engagement-count field per video, **not** individual comment text. Tier differences are descriptive, not causal.

## Local files and setup

**Purpose:** Confirm which public files are actually present on this computer. File presence does not imply that raw media or comment rows are included.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'downloaded_datasets' / 'tides_influencer'
print('Local data folder:', DATA.resolve())
if DATA.exists():
    display(pd.DataFrame([{'file': p.name, 'size_MB': round(p.stat().st_size / 1e6, 2)}
                          for p in sorted(DATA.iterdir()) if p.is_file()]))
else:
    print('Local data folder is missing. Download the files first.')

Local data folder: /Users/wang/Documents/tiktok/downloaded_datasets/tides_influencer


,file,size_MB
0,ADV_user_features.csv,1.92
1,AU_user_features.csv,21.04
2,CapEmoticon.csv,4.13
3,RGB_user_features.csv,11.65
4,demog_user_features.csv,3.81
5,emot_user_features.csv,0.27
6,tiktok_influencers.csv,0.89
7,tiktok_videos-1.csv,96.76
8,tiktok_videos-2.csv,97.62


## 1. Structure and paper/release counts

**Purpose:** Compare the public CSV release with the paper snapshot and test ID uniqueness.

In [2]:
import numpy as np
users = pd.read_csv(DATA / 'tiktok_influencers.csv', dtype={'id': str})
parts = [pd.read_csv(DATA / f'tiktok_videos-{j}.csv', dtype={'id': str, 'authorId': str}, low_memory=False)
         for j in [1, 2]]
videos = pd.concat(parts, ignore_index=True)
display(pd.DataFrame([{'creator_rows': len(users), 'unique_creator_ids': users.id.nunique(),
                       'video_rows': len(videos), 'unique_video_ids': videos.id.nunique(),
                       'paper_creators': 5000, 'paper_videos': 230406,
                       'release_minus_paper_video_rows': len(videos) - 230406}]))
print('Video authors found in creator table:', len(set(videos.authorId) & set(users.id)))
print('Released video columns:', len(videos.columns))

,creator_rows,unique_creator_ids,video_rows,unique_video_ids,paper_creators,paper_videos,release_minus_paper_video_rows
0,5000,5000,233994,233994,5000,230406,3588


Video authors found in creator table: 5000
Released video columns: 52


**What this shows:** The release has 233,994 unique video IDs, 3,588 above the paper's 230,406-video snapshot. All 5,000 creators join. Do not delete rows simply to force the paper and release counts to match.

## 2. Tier sizes

**Purpose:** Check whether equal numbers of creators also mean equal numbers of videos.

In [3]:
user_tiers = users.tier.value_counts()
video_tiers = videos.authorTier.value_counts()
display(pd.DataFrame({'creators': user_tiers, 'videos': video_tiers,
                      'mean_videos_per_creator': video_tiers / user_tiers}).sort_index().round(2))

,creators,videos,mean_videos_per_creator
macro,1000,49144,49.14
mega,1000,49774,49.77
micro,1000,46422,46.42
mid,1000,49246,49.25
nano,1000,39408,39.41


**What this shows:** Each of the five tiers has 1,000 creators, but the video totals differ. A pooled video statistic gives more weight to prolific creators.

## 3. Comments per video

**Purpose:** Compare the mean, median, zero share, and upper tail of the released comment-count snapshot.

In [4]:
for col in ['comments', 'views', 'likes', 'shares', 'duration']:
    videos[col] = pd.to_numeric(videos[col], errors='coerce')
summary = videos.groupby('authorTier')['comments'].agg(
    video_count='count', mean_comments='mean', median_comments='median',
    zero_comment_share_percent=lambda x: 100 * x.eq(0).mean(),
    p90_comments=lambda x: x.quantile(.9))
display(summary.sort_index().round(2))
print('All-video mean comments:', round(videos.comments.mean(), 2),
      'median comments:', videos.comments.median())

,video_count,mean_comments,median_comments,zero_comment_share_percent,p90_comments
authorTier,,,,,
macro,49144,201.73,31.0,3.52,331.0
mega,49774,621.88,88.0,1.70,1070.0
micro,46422,33.50,6.0,12.40,56.0
mid,49246,82.30,14.0,7.28,136.0
nano,39408,18.05,3.0,26.13,25.0


All-video mean comments: 201.66 median comments: 16.0


**What this shows:** The all-video mean is 201.66 comments, but the median is only 16. Tier medians are 3, 6, 14, 31, and 88 (Nano to Mega). A small highly engaged tail raises the mean.

## 4. Creator-level comparison

**Purpose:** Give each creator one summary instead of allowing prolific creators to dominate the tier comparison.

In [5]:
by_author = videos.groupby(['authorId', 'authorTier']).agg(
    videos_published=('id', 'size'), creator_median_comments=('comments', 'median'),
    creator_median_views=('views', 'median')).reset_index()
display(by_author.groupby('authorTier').agg(
    creators=('authorId', 'nunique'), median_videos_per_creator=('videos_published', 'median'),
    median_of_creator_comment_medians=('creator_median_comments', 'median'),
    median_of_creator_view_medians=('creator_median_views', 'median')).round(2))

,creators,median_videos_per_creator,median_of_creator_comment_medians,median_of_creator_view_medians
authorTier,,,,
macro,1000,50.0,30.5,42325.00
mega,1000,50.0,85.5,156650.00
micro,1000,50.0,6.0,3852.75
mid,1000,50.0,13.0,13175.00
nano,1000,50.0,3.0,1628.00


**What this shows:** The medians of creator-level comment medians are 3, 6, 13, 30.5, and 85.5. The tier pattern remains descriptive; it cannot prove how to gain followers.

## 5. Duration and engagement

**Purpose:** Describe duration bands and rank correlations without claiming causation.

In [6]:
bands = pd.cut(videos.duration, bins=[-1, 15, 30, 60, 180, float('inf')],
               labels=['0-15 s', '16-30 s', '31-60 s', '61-180 s', '>180 s'])
display(videos.groupby(bands, observed=True)['comments'].agg(
    video_count='count', median_comments='median', mean_comments='mean').round(2))
display(videos[['comments', 'views', 'likes', 'shares', 'duration']]
        .corr(method='spearman').round(3))

,video_count,median_comments,mean_comments
duration,,,
0-15 s,125994,10.0,142.21
16-30 s,43762,18.0,207.61
31-60 s,39884,28.0,276.84
61-180 s,22066,39.0,385.67
>180 s,2288,42.0,276.08


,comments,views,likes,shares,duration
comments,1.000,0.796,0.838,0.745,0.225
views,0.796,1.000,0.957,0.833,0.189
likes,0.838,0.957,1.000,0.826,0.169
shares,0.745,0.833,0.826,1.000,0.197
duration,0.225,0.189,0.169,0.197,1.000


**What this shows:** The comments–likes Spearman coefficient is about 0.838 and comments–views is about 0.796. Duration-band comment medians rise from 10 to 42. Author size, topic, and timing may confound these associations.

## 6. Released content and missingness

**Purpose:** Check whether media URLs and derived multimodal fields are populated.

In [7]:
columns = ['caption', 'hashtags', 'videoURL', 'musicName', 'musicURL',
           'danceability', 'video_hasText', 'comments', 'authorTier']
display(pd.DataFrame({'field': columns,
                      'non_missing_rows': [int(videos[x].notna().sum()) for x in columns],
                      'non_missing_share_percent': [100 * videos[x].notna().mean() for x in columns]}).round(2))
print('No raw MP4 or individual comment-text files are included in these local public CSVs.')

,field,non_missing_rows,non_missing_share_percent
0,caption,211043,90.19
1,hashtags,144264,61.65
2,videoURL,233994,100.00
3,musicName,233385,99.74
4,musicURL,229652,98.14
5,danceability,33326,14.24
6,video_hasText,233994,100.00
7,comments,233994,100.00
8,authorTier,233994,100.00


No raw MP4 or individual comment-text files are included in these local public CSVs.


**What this shows:** Every row has a `videoURL` value, but that does not prove the link still works. Spotify `danceability` is present for only 14.24% of videos; comment counts are not comment rows.

## Findings and limits

- The creator table contains **5,000** creators, 1,000 in each tier. The public CSVs contain **233,994** unique videos, versus **230,406** reported in the paper; both snapshots are retained.
- Per-video comment-count mean is **201.66** and median **16**. Tier medians are **3, 6, 14, 31, 88**; author-first summaries give **3, 6, 13, 30.5, 85.5**.
- Duration and engagement relationships are associations only; this sample cannot establish that longer videos cause more comments.
- Spotify `danceability` is available in only **14.24%** of released video rows. The public CSVs have comment counts, not individual comments or raw MP4s.